# GQA（分组查询注意力）教程

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/gqa_tutorial.ipynb)

本教程在 [KV Cache 教程](kv_cache_tutorial.ipynb) 和 [RoPE 教程](rope_tutorial.ipynb) 的基础上，深入介绍 **GQA**（Grouped Query Attention，分组查询注意力）。

GQA 是 LLaMA-2/3、Mistral、Falcon、Qwen 等现代 LLM 的标准注意力机制，核心目标：
在保持 MHA（多头注意力）模型质量的前提下，**大幅减少 KV Cache 内存**，从而支持更大的 batch size 或更长的上下文。

## 目录
1. MHA → MQA → GQA 演进背景
2. 架构对比可视化
3. GQA 数学原理与 Expand 操作
4. `CausalSelfAttentionGQA` 实现（RoPE + KV Cache）
5. `GPTDecoderGQA` 完整模型
6. 正确性验证
7. 参数量与 KV Cache 内存节省分析
8. 速度对比实验
9. 注意力权重可视化
10. 真实 LLM 配置（LLaMA-3、Mistral）
11. 总结

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from typing import Optional, Tuple, List

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── RoPE 辅助函数（来自 rope_tutorial.ipynb）────────────────────
def precompute_rope_freqs(d_k: int, max_len: int, theta: float = 10000.0):
    half = d_k // 2
    i = torch.arange(0, half, dtype=torch.float32)
    freqs = 1.0 / (theta ** (2.0 * i / d_k))
    positions = torch.arange(max_len, dtype=torch.float32)
    angles = torch.outer(positions, freqs)
    return angles.cos(), angles.sin()


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    d = x.shape[-1]
    x1, x2 = x[..., :d//2], x[..., d//2:]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

## 1. MHA → MQA → GQA 演进背景

### Multi-Head Attention（MHA，原始 Transformer）

每个注意力头都有**独立**的 Query、Key、Value 投影：
- $H$ 个 Query 头，$H$ 个 Key 头，$H$ 个 Value 头
- KV Cache 大小 ∝ $H \times d_k \times L$（$L$ = 序列长度）

### Multi-Query Attention（MQA，Shazeer 2019）

所有 Query 头共享**同一组** Key, Value：
- $H$ 个 Query 头，但只有 **1 个** Key 头和 1 个 Value 头
- KV Cache 降为 MHA 的 $\frac{1}{H}$
- 缺点：模型质量有所下降（所有头看同样的 Key/Value，表达能力受限）

### Grouped Query Attention（GQA，Ainslie et al. 2023）

MHA 与 MQA 的折中：将 $H$ 个 Query 头分成 $G$ 组，同组内共享 Key, Value：
- $H$ 个 Query 头，$G$ 个 Key/Value 组（$1 \le G \le H$）
- KV Cache 降为 MHA 的 $\frac{G}{H}$
- $G = H$：退化为 MHA；$G = 1$：退化为 MQA
- 实践中取 $G = H/4$ 或 $G = H/8$，在质量和效率间取得最佳平衡

| 机制 | Q 头数 | KV 组数 | KV Cache 大小 | 代表模型 |
|------|--------|---------|--------------|----------|
| MHA  | H      | H       | $H \times d_k \times L$ | GPT-2, BERT, LLaMA-1 |
| GQA  | H      | G（1<G<H）| $G \times d_k \times L$ | LLaMA-2/3, Mistral, Qwen |
| MQA  | H      | 1       | $1 \times d_k \times L$ | PaLM（部分层）|

In [ ]:
# ── MHA / GQA / MQA 架构对比可视化 ────────────────────────────
def draw_attention_arch(ax, title, H, G, q_color='#4C72B0', kv_color='#DD8452'):
    group_size = H // G
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.1)
    ax.axis('off')

    q_y, kv_y = 0.78, 0.22
    q_xs = [(i + 0.5) / H for i in range(H)]
    kv_xs = [(g + 0.5) / G for g in range(G)]

    # Draw connection lines first (behind circles)
    for i, qx in enumerate(q_xs):
        g = i // group_size
        kvx = kv_xs[g]
        ax.plot([qx, kvx], [q_y - 0.055, kv_y + 0.065],
                color='gray', linewidth=0.8, alpha=0.5, zorder=1)

    # Draw Q heads
    for i, qx in enumerate(q_xs):
        circ = plt.Circle((qx, q_y), 0.042, color=q_color, zorder=2, ec='white', lw=1)
        ax.add_patch(circ)
        ax.text(qx, q_y, f'Q{i}', ha='center', va='center',
                fontsize=max(5, 9 - H//4), color='white', fontweight='bold', zorder=3)

    # Draw KV groups
    kv_w = 0.72 / G
    for g, kvx in enumerate(kv_xs):
        rect = mpatches.FancyBboxPatch(
            (kvx - kv_w/2, kv_y - 0.065), kv_w, 0.13,
            boxstyle='round,pad=0.005', color=kv_color, alpha=0.85, zorder=2, ec='white', lw=1)
        ax.add_patch(rect)
        fs = max(5, 9 - G//2)
        ax.text(kvx, kv_y + 0.015, f'K{g}', ha='center', va='center',
                fontsize=fs, color='white', fontweight='bold', zorder=3)
        ax.text(kvx, kv_y - 0.03, f'V{g}', ha='center', va='center',
                fontsize=fs, color='white', fontweight='bold', zorder=3)

    # Labels
    ax.text(0.5, 0.97, title, ha='center', va='top',
            fontsize=13, fontweight='bold', color='#2d2d2d')
    ax.text(0.5, 0.89, f'Q heads = {H}   KV groups = {G}   group_size = {group_size}',
            ha='center', va='top', fontsize=9, color='gray')
    kv_ratio = G / H
    ax.text(0.5, 0.06,
            f'KV Cache = {kv_ratio:.0%} of MHA',
            ha='center', va='bottom', fontsize=10,
            color='#c0392b' if kv_ratio < 1 else '#27ae60', fontweight='bold')

    ax.text(-0.03, q_y,  'Q', ha='right', va='center', fontsize=10, color=q_color, fontweight='bold')
    ax.text(-0.03, kv_y, 'KV', ha='right', va='center', fontsize=10, color=kv_color, fontweight='bold')


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
H = 8
draw_attention_arch(axes[0], 'MHA  (Multi-Head Attention)',   H=H, G=H)
draw_attention_arch(axes[1], 'GQA  (Grouped Query Attention)', H=H, G=2)
draw_attention_arch(axes[2], 'MQA  (Multi-Query Attention)',   H=H, G=1)

plt.suptitle(f'Attention Architecture Comparison  (num_heads={H})',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('gqa_arch_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

## 2. GQA 数学原理与 Expand 操作

### 参数定义

- $H$：Query 头数（`num_heads`）
- $G$：KV 组数（`num_kv_heads`），$G$ 能整除 $H$
- $g = H/G$：每组包含的 Query 头数（`group_size`）

### 投影矩阵

$$W_Q \in \mathbb{R}^{d_{model} \times (H \cdot d_k)}, \quad W_K \in \mathbb{R}^{d_{model} \times (G \cdot d_k)}, \quad W_V \in \mathbb{R}^{d_{model} \times (G \cdot d_k)}$$

相比 MHA，$W_K$ 和 $W_V$ 各缩小 $G/H$ 倍。

### Expand 操作（核心步骤）

第 $i$ 个 Query 头属于第 $\lfloor i/g \rfloor$ 组，使用该组的 Key/Value：

$$\text{head}_i = \text{Attention}(Q_i,\; K_{\lfloor i/g \rfloor},\; V_{\lfloor i/g \rfloor})$$

实现时通过 `repeat_interleave` 将 KV 从 $G$ 组扩展到 $H$ 头：

```
K shape:  (B, G, L, d_k)
  └─ repeat_interleave(group_size, dim=1)
K_exp:    (B, H, L, d_k)   ← [K0,K0,...,K0, K1,K1,...,K1, ...]
                                 |←group_size→|  |←group_size→|
```

### 内存高效替代（expand，无数据拷贝）

```python
K_exp = K.unsqueeze(2)                              # (B, G, 1, L, d_k)
K_exp = K_exp.expand(B, G, group_size, L, d_k)     # (B, G, g, L, d_k)  — 共享内存
K_exp = K_exp.reshape(B, H, L, d_k)                # (B, H, L, d_k)
```

In [ ]:
class CausalSelfAttentionGQA(nn.Module):
    """
    Grouped Query Attention (GQA) with RoPE + KV Cache

    num_heads    (H): Query 头数
    num_kv_heads (G): Key/Value 组数，H 必须能被 G 整除

    特殊情形：
      G = H → Multi-Head Attention (MHA)
      G = 1 → Multi-Query Attention (MQA)
      1 < G < H → GQA
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_kv_heads: int,
        max_len: int = 4096,
        theta: float = 10000.0,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model 必须能被 num_heads 整除'
        assert num_heads % num_kv_heads == 0, 'num_heads 必须能被 num_kv_heads 整除'

        self.d_model      = d_model
        self.num_heads    = num_heads       # H
        self.num_kv_heads = num_kv_heads    # G
        self.group_size   = num_heads // num_kv_heads   # g = H/G
        self.d_k          = d_model // num_heads

        # Q: 全量 H 头；K, V: 只有 G 组 ← 参数量节省
        self.W_q = nn.Linear(d_model, num_heads    * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)  # 更小
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)  # 更小
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.last_attn_weights = None

        # RoPE 频率表（d_k 维，每个头独立旋转）
        cos_table, sin_table = precompute_rope_freqs(self.d_k, max_len, theta)
        self.register_buffer('rope_cos', cos_table)
        self.register_buffer('rope_sin', sin_table)

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        x      : (B, T, d_model)
        past_kv: (K_cache, V_cache)
                  K_cache: (B, G, past_len, d_k)  ← G 不是 H！
                  V_cache: (B, G, past_len, d_k)
        """
        B, T, _ = x.shape
        H, G, g = self.num_heads, self.num_kv_heads, self.group_size

        # ── Q: (B, H, T, d_k)；K, V: (B, G, T, d_k) ─────────
        Q = self.W_q(x).view(B, T, H, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, G, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, G, self.d_k).transpose(1, 2)

        # ── RoPE（Q 和 K 均旋转，V 不旋转）───────────────────
        past_len = past_kv[0].size(2) if past_kv is not None else 0
        cos = self.rope_cos[past_len : past_len + T]
        sin = self.rope_sin[past_len : past_len + T]
        Q = apply_rope(Q, cos, sin)   # (B, H, T, d_k)
        K = apply_rope(K, cos, sin)   # (B, G, T, d_k)

        # ── KV Cache：缓存 G 组（远小于 MHA 的 H 组）─────────
        if past_kv is not None:
            K_cache, V_cache = past_kv
            K = torch.cat([K_cache, K], dim=2)   # (B, G, past_len+T, d_k)
            V = torch.cat([V_cache, V], dim=2)

        total_len = K.size(2)
        new_kv = (K.detach(), V.detach()) if use_cache else None

        # ── Expand K, V: G 组 → H 头 ──────────────────────────
        # repeat_interleave: [K0,K0,...,K0, K1,K1,...] (每组重复 group_size 次)
        K_exp = K.repeat_interleave(g, dim=1)  # (B, H, total_len, d_k)
        V_exp = V.repeat_interleave(g, dim=1)

        # ── 标准因果注意力 ──────────────────────────────────────
        scores = torch.matmul(Q, K_exp.transpose(-2, -1)) / math.sqrt(self.d_k)

        if T > 1:
            row = torch.arange(T, device=x.device).view(T, 1)
            col = torch.arange(total_len, device=x.device).view(1, total_len)
            causal_mask = col <= (past_len + row)
            scores = scores.masked_fill(
                ~causal_mask.unsqueeze(0).unsqueeze(0), float('-inf')
            )

        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        self.last_attn_weights = attn.detach()   # (B, H, T, total_len)

        out = torch.matmul(self.dropout(attn), V_exp)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out), new_kv


# ── 快速验证三种模式 ─────────────────────────────────────────
for name, H, G in [('MHA (G=H=8)', 8, 8), ('GQA (G=2)', 8, 2), ('MQA (G=1)', 8, 1)]:
    attn = CausalSelfAttentionGQA(d_model=64, num_heads=H, num_kv_heads=G, max_len=64)
    x = torch.randn(2, 6, 64)
    out, kv = attn(x, use_cache=True)
    print(f'{name:18s}: out={tuple(out.shape)},  K cache={tuple(kv[0].shape)}')

In [ ]:
class GPTBlockGQA(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, d_ff,
                 max_len=4096, theta=10000.0, dropout=0.1):
        super().__init__()
        self.attn = CausalSelfAttentionGQA(
            d_model, num_heads, num_kv_heads, max_len, theta, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, past_kv=None, use_cache=False):
        attn_out, new_kv = self.attn(self.norm1(x), past_kv, use_cache)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class GPTDecoderGQA(nn.Module):
    """
    GPT-style Decoder with GQA + RoPE + KV Cache

    与 GPTDecoderRoPE 相比：
      - num_kv_heads 参数控制 KV 组数（< num_heads 即启用 GQA）
      - KV Cache 每层只存 num_kv_heads 组，内存大幅节省
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        num_kv_heads: int,
        num_layers: int,
        d_ff: int,
        max_len: int = 4096,
        theta: float = 10000.0,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.max_len = max_len
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPTBlockGQA(d_model, num_heads, num_kv_heads, d_ff, max_len, theta, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, past_kv_list=None, use_cache=False):
        x = self.drop(self.token_emb(idx))
        new_kv_list = []
        for i, block in enumerate(self.blocks):
            past_kv = past_kv_list[i] if past_kv_list is not None else None
            x, new_kv = block(x, past_kv=past_kv, use_cache=use_cache)
            new_kv_list.append(new_kv)
        logits = self.lm_head(self.norm(x))
        return logits, (new_kv_list if use_cache else None)

    @torch.no_grad()
    def generate_no_cache(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_len:])
            next_token = logits[:, -1, :].argmax(-1, keepdim=True)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

    @torch.no_grad()
    def generate_with_cache(self, idx, max_new_tokens):
        logits, kv = self(idx, use_cache=True)
        next_token = logits[:, -1, :].argmax(-1, keepdim=True)
        idx = torch.cat([idx, next_token], dim=1)
        for _ in range(max_new_tokens - 1):
            logits, kv = self(next_token, past_kv_list=kv, use_cache=True)
            next_token = logits[:, -1, :].argmax(-1, keepdim=True)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# ── 参数量对比（不同 num_kv_heads）──────────────────────────
base_cfg = dict(vocab_size=1000, d_model=256, num_heads=8,
                num_layers=4, d_ff=1024, max_len=512)

print(f'  {"Config":25s}  {"Params":>12s}  {"KV Cache (seq=512, L=4)"}\n' + '-'*68)
for name, G in [("MHA (G=8)", 8), ("GQA (G=4)", 4), ("GQA (G=2)", 2), ("MQA (G=1)", 1)]:
    m = GPTDecoderGQA(**base_cfg, num_kv_heads=G).to(DEVICE)
    d_k = base_cfg['d_model'] // base_cfg['num_heads']
    kv_mb = (base_cfg['num_layers'] * 2 * G * d_k * 512 * 2) / (1024**2)
    print(f'  {name:25s}  {count_params(m):>12,}  {kv_mb:.2f} MB')

## 3. 正确性验证

验证两个核心等价性：
1. `generate_no_cache` == `generate_with_cache`（GQA + KV Cache 正确性）
2. GQA 当 `num_kv_heads = num_heads` 时，行为与 MHA 完全等价

In [ ]:
torch.manual_seed(99)

# ── 验证 1：no_cache == with_cache（GQA 模式 G=2）─────────────
model_gqa = GPTDecoderGQA(
    vocab_size=200, d_model=64, num_heads=8, num_kv_heads=2,
    num_layers=2, d_ff=256, max_len=128
).to(DEVICE).eval()

prompt = torch.randint(1, 200, (1, 10)).to(DEVICE)
N_NEW = 8

with torch.no_grad():
    out_no  = model_gqa.generate_no_cache(prompt.clone(), N_NEW)
    out_kv  = model_gqa.generate_with_cache(prompt.clone(), N_NEW)

match1 = torch.all(out_no == out_kv).item()
print('验证 1 — GQA (G=2): no_cache == with_cache ?', '✓' if match1 else '✗')

# ── 验证 2：GQA(G=H) 与 MHA 等价（权重相同时输出相同）─────────
# GQA 当 G=H 时，每组只有 1 个 Query 头，repeat_interleave(1) = identity
# → 与标准 MHA 在数学上完全等价
torch.manual_seed(7)

model_mha_via_gqa = GPTDecoderGQA(
    vocab_size=100, d_model=64, num_heads=4, num_kv_heads=4,  # G=H → MHA
    num_layers=2, d_ff=256, max_len=64
).to(DEVICE).eval()

p2 = torch.randint(1, 100, (1, 8)).to(DEVICE)
with torch.no_grad():
    out_no2 = model_mha_via_gqa.generate_no_cache(p2.clone(), 6)
    out_kv2 = model_mha_via_gqa.generate_with_cache(p2.clone(), 6)

match2 = torch.all(out_no2 == out_kv2).item()
print('验证 2 — GQA (G=H, MHA 模式): no_cache == with_cache ?', '✓' if match2 else '✗')

# ── 验证 3：MQA(G=1) 正确性 ──────────────────────────────────
model_mqa = GPTDecoderGQA(
    vocab_size=200, d_model=64, num_heads=8, num_kv_heads=1,  # MQA
    num_layers=2, d_ff=256, max_len=128
).to(DEVICE).eval()

with torch.no_grad():
    out_no3 = model_mqa.generate_no_cache(prompt.clone(), N_NEW)
    out_kv3 = model_mqa.generate_with_cache(prompt.clone(), N_NEW)

match3 = torch.all(out_no3 == out_kv3).item()
print('验证 3 — MQA (G=1): no_cache == with_cache ?', '✓' if match3 else '✗')

print(f'\n所有正确性验证通过: {all([match1, match2, match3])} ✓')

## 4. 参数量与 KV Cache 内存节省分析

### 参数量

注意力层参数量：
$$\text{Params}_{\text{attn}} = d_{model}^2 \cdot \left(1 + \frac{2G}{H} + 1\right) = d_{model}^2 \cdot \left(2 + \frac{2G}{H}\right)$$

与 MHA（$G=H$）相比，每层节省 $d_{model}^2 \cdot 2 \cdot (1 - G/H)$ 个参数。

### KV Cache 内存

$$\text{KV Cache} = L \times 2 \times B \times G \times d_k \times S \times \text{bytes}$$

相比 MHA（$G = H$），KV Cache 缩小为原来的 $G/H$。

这是 GQA 最核心的收益：序列越长，节省越显著。

In [ ]:
# ── 参数量 & 内存节省定量分析 ──────────────────────────────────

def attn_params(d_model, H, G):
    d_k = d_model // H
    W_q = d_model * H * d_k
    W_k = d_model * G * d_k
    W_v = d_model * G * d_k
    W_o = d_model * d_model
    return W_q + W_k + W_v + W_o


def kv_cache_mb(G, d_k, seq_len, num_layers, batch=1, dtype_bytes=2):
    return (num_layers * 2 * batch * G * d_k * seq_len * dtype_bytes) / (1024**2)


# ── 图1：参数量 & KV Cache 节省比率 ──────────────────────────
d_model, H, L = 4096, 32, 32
d_k = d_model // H

G_values = [H, H//2, H//4, H//8, H//16, 1]
G_labels = [f'G={g}\n({"MHA" if g==H else "MQA" if g==1 else "GQA"})' for g in G_values]

param_ratios = [attn_params(d_model, H, G) / attn_params(d_model, H, H) for G in G_values]
kv_ratios    = [G / H for G in G_values]

seq_lengths = [512, 1024, 2048, 4096, 8192]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 子图1：参数量比率
ax = axes[0]
bars1 = ax.bar(range(len(G_values)), param_ratios, color='steelblue', edgecolor='k', linewidth=0.5)
ax.set_xticks(range(len(G_values)))
ax.set_xticklabels(G_labels, fontsize=8)
ax.set_ylabel('Attention Params / MHA Params')
ax.set_title(f'Attention 参数量比率\n(d_model={d_model}, H={H})', fontsize=11)
ax.axhline(1.0, color='red', linestyle='--', linewidth=1, alpha=0.6)
ax.set_ylim(0, 1.15)
for bar, ratio in zip(bars1, param_ratios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{ratio:.0%}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# 子图2：KV Cache 比率
ax2 = axes[1]
bars2 = ax2.bar(range(len(G_values)), kv_ratios,
                color=['#c0392b' if r==1 else '#e67e22' if r>0.25 else '#27ae60' for r in kv_ratios],
                edgecolor='k', linewidth=0.5)
ax2.set_xticks(range(len(G_values)))
ax2.set_xticklabels(G_labels, fontsize=8)
ax2.set_ylabel('KV Cache Size / MHA KV Cache')
ax2.set_title(f'KV Cache 大小比率\n(关键收益：越小越好)', fontsize=11)
for bar, ratio in zip(bars2, kv_ratios):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{ratio:.0%}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.grid(True, axis='y', alpha=0.3)

# 子图3：绝对内存（不同 seq 长度）
ax3 = axes[2]
for G, style, color in zip([H, H//4, H//8, 1],
                            ['-o', '-s', '-^', '-D'],
                            ['#c0392b', '#e67e22', '#27ae60', '#2980b9']):
    label = f'MHA (G={H})' if G==H else f'MQA (G=1)' if G==1 else f'GQA (G={G})'
    mems = [kv_cache_mb(G, d_k, s, L) for s in seq_lengths]
    ax3.plot(seq_lengths, mems, style, color=color, linewidth=2, markersize=6, label=label)
ax3.set_xlabel('Sequence Length', fontsize=11)
ax3.set_ylabel('KV Cache (MB, float16, L=32, B=1)')
ax3.set_title(f'KV Cache 绝对内存 vs 序列长度\n(d_model={d_model}, H={H})', fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.suptitle('GQA 参数量 & KV Cache 节省分析', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gqa_memory_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n示例（d_model={d_model}, H={H}, L={L}, seq=2048, float16）:')
print(f'  {"Config":20s}  {"Attn Params":>15s}  {"KV Cache (MB)":>15s}')
print('-' * 55)
for G in G_values:
    label = f'MHA (G={H})' if G==H else f'MQA (G=1)' if G==1 else f'GQA (G={G})'
    p = attn_params(d_model, H, G) * L
    m = kv_cache_mb(G, d_k, 2048, L)
    print(f'  {label:20s}  {p:>15,}  {m:>15.1f}')

## 5. 速度对比实验

GQA 的主要加速来源：
1. **KV 投影计算量减少**：$W_K, W_V$ 更小，每步计算量为 MHA 的 $G/H$
2. **内存带宽节省**：KV Cache 更小，读写更快（GPU 上效果更显著）
3. **更大 batch size 支持**：相同显存可容纳更多请求

In [ ]:
# ── 速度基准测试 ──────────────────────────────────────────────
BENCH_CFG = dict(
    vocab_size=2000, d_model=256, num_heads=8,
    num_layers=4, d_ff=1024, max_len=512
)
PROMPT_LEN = 32
GEN_LENS   = [16, 32, 64, 128]
N_RUNS     = 3

models = {}
for name, G in [('MHA (G=8)', 8), ('GQA (G=4)', 4), ('GQA (G=2)', 2), ('MQA (G=1)', 1)]:
    m = GPTDecoderGQA(**BENCH_CFG, num_kv_heads=G).to(DEVICE).eval()
    models[name] = m

prompt_bench = torch.randint(1, 2000, (1, PROMPT_LEN)).to(DEVICE)


def measure(fn, n=N_RUNS):
    with torch.no_grad():
        fn()  # warmup
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n): fn()
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n * 1000  # ms


results = {name: [] for name in models}
print(f'  {"Config":15s}', end='')
for g in GEN_LENS: print(f'  gen={g:3d}ms', end='')
print()
print('-' * (16 + len(GEN_LENS) * 12))

for name, model in models.items():
    row = []
    print(f'  {name:15s}', end='', flush=True)
    for gen_len in GEN_LENS:
        t = measure(lambda: model.generate_with_cache(prompt_bench.clone(), gen_len))
        row.append(t)
        print(f'  {t:8.1f}', end='', flush=True)
    results[name] = row
    print()

# ── 可视化 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#c0392b', '#e67e22', '#27ae60', '#2980b9']
markers = ['o', 's', '^', 'D']

ax = axes[0]
for (name, times), c, mk in zip(results.items(), colors, markers):
    ax.plot(GEN_LENS, times, f'-{mk}', color=c, linewidth=2, markersize=7, label=name)
ax.set_xlabel('Generated Tokens', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Generation Time (with KV Cache)', fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax2 = axes[1]
mha_times = results['MHA (G=8)']
for (name, times), c, mk in zip(results.items(), colors, markers):
    speedup = [mha / t for mha, t in zip(mha_times, times)]
    ax2.plot(GEN_LENS, speedup, f'-{mk}', color=c, linewidth=2, markersize=7, label=name)
ax2.axhline(1.0, color='gray', linestyle='--', linewidth=1)
ax2.set_xlabel('Generated Tokens', fontsize=12)
ax2.set_ylabel('Speedup vs MHA', fontsize=12)
ax2.set_title('Speedup Relative to MHA', fontsize=12)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.suptitle(f'GQA Speed Benchmark  (d_model={BENCH_CFG["d_model"]}, '
             f'H={BENCH_CFG["num_heads"]}, L={BENCH_CFG["num_layers"]})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gqa_speed_benchmark.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. 注意力权重可视化

GQA 的一个有趣特性：同一 KV 组内的多个 Query 头会产生**不同**的注意力分布，
因为它们的 Q 向量不同（只有 K, V 共享）。

下面可视化同一组内不同 Q 头的注意力模式，以及跨组对比。

In [ ]:
torch.manual_seed(0)

H_VIS, G_VIS = 8, 2   # GQA: 8 Q heads, 2 KV groups, group_size=4
D_VIS, SEQ_VIS = 64, 10

vis_model = GPTDecoderGQA(
    vocab_size=100, d_model=D_VIS, num_heads=H_VIS, num_kv_heads=G_VIS,
    num_layers=1, d_ff=256, max_len=64, dropout=0.0
).to(DEVICE).eval()

sample_ids = torch.randint(1, 100, (1, SEQ_VIS)).to(DEVICE)
with torch.no_grad():
    logits, kv = vis_model(sample_ids, use_cache=True)

# attn_weights: (1, H, SEQ, SEQ)
attn_w = vis_model.blocks[0].attn.last_attn_weights[0].cpu().numpy()  # (H, SEQ, SEQ)

group_size = H_VIS // G_VIS   # 4
token_labels = [f't{i}' for i in range(SEQ_VIS)]

fig, axes = plt.subplots(G_VIS, group_size, figsize=(14, 6))

for g in range(G_VIS):
    for j in range(group_size):
        head_idx = g * group_size + j
        ax = axes[g][j]
        sns.heatmap(attn_w[head_idx], ax=ax, cmap='Blues', vmin=0, vmax=1,
                    xticklabels=token_labels, yticklabels=token_labels,
                    linewidths=0.3, linecolor='lightgray', cbar=(j == group_size-1))
        ax.set_title(f'KV Group {g} / Q Head {head_idx}', fontsize=9, fontweight='bold')
        ax.tick_params(labelsize=7)
        if j == 0: ax.set_ylabel(f'KV Group {g}', fontsize=9)
        else:      ax.set_ylabel('')
        ax.set_xlabel('Key')

plt.suptitle(f'GQA 注意力权重可视化 (H={H_VIS}, G={G_VIS}, group_size={group_size})\n'
             f'同一 KV 组内各 Q 头产生不同分布（Q 向量不同），但共享相同的 K/V',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('gqa_attn_weights.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 同组头之间的注意力相关性 ──────────────────────────────────
fig2, ax = plt.subplots(figsize=(7, 5))
# flatten attention maps for each head: (H, SEQ*SEQ)
attn_flat = attn_w.reshape(H_VIS, -1)   # (8, 100)
corr_matrix = np.corrcoef(attn_flat)     # (8, 8)

sns.heatmap(corr_matrix, ax=ax, cmap='RdBu', vmin=-1, vmax=1,
            xticklabels=[f'H{i}' for i in range(H_VIS)],
            yticklabels=[f'H{i}' for i in range(H_VIS)],
            linewidths=0.5, annot=True, fmt='.2f', annot_kws={'size': 8})
ax.set_title(f'各 Q 头注意力分布的相关性矩阵\n'
             f'(H0-H3 属于 KV Group 0，H4-H7 属于 KV Group 1)',
             fontsize=11)

# Add group boundary lines
for g in range(1, G_VIS):
    ax.axhline(g * group_size, color='red', linewidth=2)
    ax.axvline(g * group_size, color='red', linewidth=2)

plt.tight_layout()
plt.savefig('gqa_head_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. 真实 LLM 配置

GQA 已成为现代大语言模型的标准配置。下面列出主要模型的参数：

In [ ]:
# ── 真实 LLM 的 GQA 配置分析 ──────────────────────────────────
real_models = [
    # (name,         d_model, H,  G,  L,   context)
    ('LLaMA-1 7B',   4096,   32, 32, 32,  2048),   # MHA
    ('LLaMA-2 7B',   4096,   32, 32, 32,  4096),   # MHA
    ('LLaMA-2 70B',  8192,   64,  8, 80,  4096),   # GQA
    ('LLaMA-3 8B',   4096,   32,  8, 32,  8192),   # GQA
    ('LLaMA-3 70B',  8192,   64,  8, 80,  8192),   # GQA
    ('Mistral 7B',   4096,   32,  8, 32,  32768),  # GQA + Sliding Window
    ('Qwen-2 7B',    3584,   28,  4, 28,  131072), # GQA
    ('Falcon 40B',   8192,   64,  8, 60,  2048),   # GQA
]

print(f'{"Model":>20} {"d_model":>8} {"H":>4} {"G":>4} {"g=H/G":>6} '
      f'{"KV Ratio":>9} {"KV@8k (GB)"}\n' + '-'*78)

kv_at_8k = []
for name, d, H, G, L, ctx in real_models:
    d_k = d // H
    g = H // G
    ratio = G / H
    kv_gb = (L * 2 * 1 * G * d_k * 8192 * 2) / (1024**3)
    kv_at_8k.append(kv_gb)
    attn_type = 'MHA' if G == H else ('MQA' if G == 1 else 'GQA')
    print(f'{name:>20} {d:>8} {H:>4} {G:>4} {g:>6}  '
          f'{ratio:>8.0%}  {kv_gb:>8.2f} GB  [{attn_type}]')

# ── 可视化 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = [m[0] for m in real_models]
kv_ratios_real = [m[2]/m[3] if m[3]>0 else 1 for m in real_models]  # G/H... wait
kv_ratios_real = [m[3]/m[2] for m in real_models]  # G/H

ax = axes[0]
bar_colors = ['#c0392b' if r==1 else '#27ae60' for r in kv_ratios_real]
bars = ax.barh(names[::-1], [r for r in kv_ratios_real[::-1]],
               color=bar_colors[::-1], edgecolor='k', linewidth=0.5)
ax.axvline(1.0, color='red', linestyle='--', linewidth=1, label='MHA baseline')
ax.set_xlabel('KV Cache Ratio (G/H)', fontsize=11)
ax.set_title('KV Cache Size vs MHA\n(越短越省内存)', fontsize=11)
for bar, r in zip(bars, kv_ratios_real[::-1]):
    ax.text(r + 0.01, bar.get_y() + bar.get_height()/2,
            f'{r:.0%}', va='center', fontsize=9)
ax.grid(True, axis='x', alpha=0.3)
ax.legend()

ax2 = axes[1]
bar_colors2 = ['#c0392b' if r==1 else '#27ae60' for r in kv_ratios_real]
ax2.barh(names[::-1], kv_at_8k[::-1],
         color=bar_colors2[::-1], edgecolor='k', linewidth=0.5)
ax2.set_xlabel('KV Cache (GB, seq=8192, float16, batch=1)', fontsize=10)
ax2.set_title('绝对 KV Cache 占用\n(seq=8192, float16)', fontsize=11)
ax2.grid(True, axis='x', alpha=0.3)

plt.suptitle('真实 LLM GQA 配置对比', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gqa_real_llm_configs.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. 总结

### GQA 核心知识点

| 维度 | MHA | GQA | MQA |
|------|-----|-----|-----|
| KV 组数 | $G = H$ | $1 < G < H$ | $G = 1$ |
| KV Cache | $H \times d_k \times L$ | $G \times d_k \times L$ | $1 \times d_k \times L$ |
| 参数节省 | 无 | $2d^2(1 - G/H)$/层 | $2d^2(1 - 1/H)$/层 |
| 模型质量 | 最好 | 接近 MHA | 有所下降 |
| 典型 $G/H$ | 100% | 12.5%~25% | 3%（$H=32$） |

### 实现关键点

```python
# 1. 参数设置：W_K, W_V 更小
self.W_k = nn.Linear(d_model, num_kv_heads * d_k)   # G * d_k 而非 H * d_k
self.W_v = nn.Linear(d_model, num_kv_heads * d_k)

# 2. KV Cache：只缓存 G 组（节省 H/G 倍内存）
# K cache shape: (B, G, seq_len, d_k)  ← 远小于 MHA 的 (B, H, seq_len, d_k)

# 3. Expand：将 G 组扩展到 H 头再做注意力
K_exp = K.repeat_interleave(group_size, dim=1)   # (B, G, L, d_k) → (B, H, L, d_k)

# 4. 与 RoPE 无缝集成（RoPE 分别旋转 H 个 Q 和 G 个 K）
Q = apply_rope(Q, cos, sin)   # (B, H, T, d_k)
K = apply_rope(K, cos, sin)   # (B, G, T, d_k)  ← G 个 K，再 expand
```

### 与前两篇教程的知识衔接

```
KV Cache 教程  →  每层缓存 H 个 K/V                  O(H·d·L) 内存
RoPE 教程      →  旋转 Q 和 K 实现相对位置感知
GQA 教程（本篇）→  只缓存 G 个 K/V（G << H）          O(G·d·L) 内存
                   同组内多个 Q 头共享旋转后的 K/V
                   → LLaMA-2/3 / Mistral / Qwen 标准配置
```

### 进阶方向

- **GQA 训练技巧（Uptrained GQA）**：从 MHA 检查点蒸馏出 GQA 模型（论文原始做法）
- **Multi-Latent Attention（MLA）**：DeepSeek-V2 进一步用低秩 KV 压缩，KV Cache 更小
- **Flash Attention + GQA**：FlashAttention-2 原生支持 GQA，可直接传 `num_key_value_heads`
- **vLLM + GQA**：PagedAttention 与 GQA 结合，实现高吞吐量推理服务

### 延伸阅读

- [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245)
- [Fast Transformer Decoding: One Write-Head is All You Need (MQA)](https://arxiv.org/abs/1911.02150)
- [LLaMA 2: Open Foundation and Fine-Tuned Chat Models](https://arxiv.org/abs/2307.09288)
- [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691)